# Análise do Histórico da Simulação (FLPOPT)

Este notebook explora o histórico gerado pelas 20 rodadas da otimização, extraindo informações sobre a evolução da penalidade por não seleção (`unselected_count`), frentes de Pareto e variáveis de decisão.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Carregar o histórico de rodadas
with open('historico_20_rodadas.json', 'r') as f:
    history = json.load(f)
    
print(f"Número de rodadas salvas no histórico: {len(history)}")

## 1. Evolução do `unselected_count` ao longo das rodadas
Vamos analisar como a contagem de clientes não selecionados se acumulou rodada a rodada. A lógica incorporada ao otimizador aumenta a probabilidade/peso de escolha de um cliente quanto maior for o seu `unselected_count`.

In [ ]:
unselected_counts = []
for i, rnd in enumerate(history):
    # Extrair os valores de unselected_count que foram usados na rodada
    unselected = rnd['input']['unselected_count']
    unselected_counts.append(unselected)

df_unselected = pd.DataFrame(unselected_counts)
df_unselected.index.name = 'Rodada'

plt.figure(figsize=(12, 6))
for col in df_unselected.columns:
    plt.plot(df_unselected.index, df_unselected[col], marker='o', markersize=4, label=f'Cliente {col}')

plt.title('Evolução do unselected_count (Rodadas sem ser selecionado)', fontsize=14)
plt.xlabel('Rodada', fontsize=12)
plt.ylabel('Contagem', fontsize=12)
plt.xticks(range(len(history)))
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 2. Fronteiras de Pareto
A otimização MCDM tenta minimizar os 3 objetivos. Vamos observar a fronteira de Pareto em pontos do início, do meio e do final da simulação para compreender o impacto da evolução da rodada nas frentes obtidas.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Vamos plotar 3 momentos chave
rodadas_plot = [0, len(history)//2, len(history)-1]
colors = ['#e63946', '#2a9d8f', '#1d3557']
markers = ['o', '^', 's']

for r, c, m in zip(rodadas_plot, colors, markers):
    F = np.array(history[r]['result']['F'])
    if F is not None and len(F) > 0:
        ax.scatter(F[:, 0], F[:, 1], F[:, 2], c=c, marker=m, label=f'Rodada {r}', alpha=0.7, s=40)

ax.set_xlabel('f1 (Energia Consumida)', fontweight='bold')
ax.set_ylabel('f2 (Recompensa de Não Seleção)', fontweight='bold')
ax.set_zlabel('f3 (Tempo T global)', fontweight='bold')
ax.set_title('Evolução das Fronteiras de Pareto 3D', fontsize=15)
ax.view_init(elev=20, azim=45)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Análise das Decisões (Média de seleção)
Vamos inspecionar estatisticamente os clientes: na fronteira de Pareto de uma rodada arbitrária, qual é a proporção em que cada cliente é escolhido (variáveis `beta_n`) e como as frequências de processador (`f_n`) estão distribuídas.

In [ ]:
rodada_alvo = len(history) - 1 # Última rodada
solucoes_rodada = history[rodada_alvo]['result']['X']
if solucoes_rodada:
    df_x = pd.DataFrame(solucoes_rodada)
    
    # Extrair as colunas beta
    beta_cols = [col for col in df_x.columns if col.startswith('beta_')]
    f_cols = [col for col in df_x.columns if col.startswith('f_')]
    
    plt.figure(figsize=(15, 5))
    
    # Proporção de Seleção (betas = True)
    plt.subplot(1, 2, 1)
    proporcao_selecao = df_x[beta_cols].mean() * 100
    sns.barplot(x=[col.split('_')[1] for col in beta_cols], y=proporcao_selecao, palette="viridis")
    plt.title(f'Proporção de Seleção dos Clientes (%) - Fronteira da Rodada {rodada_alvo}')
    plt.xlabel('Cliente n')
    plt.ylabel('Frequência em que beta_n é True (%)')
    
    # Distribuição de Frequências (f_n)
    plt.subplot(1, 2, 2)
    sns.boxplot(data=df_x[f_cols], palette="crest")
    plt.title(f'Distribuição de Frequências (f_n) alocadas - Rodada {rodada_alvo}')
    plt.xlabel('Cliente n')
    plt.ylabel('Frequência f_n')
    plt.xticks(range(len(f_cols)), [col.split('_')[1] for col in f_cols])
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Nenhuma solução salva na rodada {rodada_alvo}.")

## 4. Análise da Variável Global de Tempo (T)
Observando como o Tempo global (variável `T`) se distribui nas soluções e evolui ao longo das rodadas.

In [ ]:
t_means = []
t_mins = []
t_maxs = []

for rnd in history:
    if rnd['result']['X']:
        df_tmp = pd.DataFrame(rnd['result']['X'])
        t_means.append(df_tmp['T'].mean())
        t_mins.append(df_tmp['T'].min())
        t_maxs.append(df_tmp['T'].max())
    else:
        t_means.append(np.nan)
        t_mins.append(np.nan)
        t_maxs.append(np.nan)

plt.figure(figsize=(14, 5))

# Histograma de T na rodada alvo
plt.subplot(1, 2, 1)
if solucoes_rodada:
    sns.histplot(df_x['T'], bins=20, kde=True, color='purple')
    plt.title(f'Distribuição do Tempo (T) - Rodada {rodada_alvo}')
    plt.xlabel('Tempo Global (T)')
    plt.ylabel('Frequência de Soluções')

# Evolução de T ao longo das rodadas
plt.subplot(1, 2, 2)
rodadas = range(len(history))
plt.plot(rodadas, t_means, label='Média de T (Fronteira)', color='purple', marker='o')
plt.fill_between(rodadas, t_mins, t_maxs, color='purple', alpha=0.2, label='Mínimo e Máximo')
plt.title('Evolução do Tempo Global T ao longo das Rodadas')
plt.xlabel('Rodada')
plt.ylabel('Tempo (T)')
plt.xticks(rodadas)
plt.legend()

plt.tight_layout()
plt.show()

## 5. Análise Isolada das Frequências (f_n)
Verificando a evolução das frequências médias `f_n` atribuídas a cada cliente nas fronteiras de Pareto geradas a cada rodada.

In [ ]:
f_means_per_round = []
for rnd in history:
    if rnd['result']['X']:
        df_tmp = pd.DataFrame(rnd['result']['X'])
        f_cols_tmp = [col for col in df_tmp.columns if col.startswith('f_')]
        f_means_per_round.append(df_tmp[f_cols_tmp].mean().values)
    else:
        f_means_per_round.append(np.full(len(f_cols), np.nan))

df_f_evo = pd.DataFrame(f_means_per_round, columns=[col.split('_')[1] for col in f_cols])
df_f_evo.index.name = 'Rodada'

plt.figure(figsize=(12, 6))
sns.heatmap(df_f_evo.T, cmap='YlGnBu', annot=False, cbar_kws={'label': 'Frequência Média (f_n)'})
plt.title('Mapa de Calor: Frequência Média (f_n) por Cliente ao Longo das Rodadas', fontsize=14)
plt.xlabel('Rodada', fontsize=12)
plt.ylabel('Cliente n', fontsize=12)
plt.tight_layout()
plt.show()